# ScientificLoop: GRPO Training Notebook

> **Teaching an LLM to reproduce scientific papers through reinforcement learning.**

This notebook trains **Qwen2.5-Coder-7B-Instruct** with **GRPO** (Group Relative Policy Optimization) via LoRA on the ScientificLoop RL environment.

The agent reads ML paper methodology, generates Python code, executes it in a sandbox, and receives reward based on how closely the measured results match the paper's reported values.

**Runtime required:** GPU (T4 minimum, A10G recommended)  
**Estimated time:** ~2.5 hours on A10G, ~4+ hours on T4  
**VRAM required:** ~22GB (use gradient checkpointing + LoRA)

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sushant0809/scientific-loop/blob/main/train_grpo.ipynb)

## Step 1: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch runtime to GPU')

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install -q \
    "trl>=1.2.0" \
    transformers \
    accelerate \
    datasets \
    peft \
    "openenv-scientific-loop @ git+https://huggingface.co/spaces/Sushant0809/scientific-loop"

print('Dependencies installed.')

## Step 3: Authenticate with HuggingFace

Needed to push the trained LoRA adapter to your Hub repo. Set `push_to_hub=False` in the config below if you want to skip this.

In [ ]:
from huggingface_hub import login
# Either paste your token directly or use the interactive login:
# login()  
import os
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # set as Colab secret or paste here
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in.')
else:
    print('No HF_TOKEN found — push_to_hub will be disabled.')

## Step 4: Configuration

In [ ]:
import os

# ── Change these to match your setup ─────────────────────────────────────────
MODEL_NAME     = 'Qwen/Qwen2.5-Coder-7B-Instruct'   # or 1.5B for T4
OUTPUT_DIR     = './outputs/scientific-loop-grpo'
HUB_MODEL_ID   = 'Sushant0809/scientific-loop-grpo'  # your HF repo
PUSH_TO_HUB    = bool(HF_TOKEN)                      # disable if no token
TOTAL_EPISODES = 200
NUM_EPOCHS     = 2      # 2 epochs = 100 steps ≈ 2.5hr on A10G

# Fix CUDA memory fragmentation before torch import
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model: {MODEL_NAME}  |  Output: {OUTPUT_DIR}  |  Push: {PUSH_TO_HUB}')

## Step 5: Load Model + Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model (bfloat16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
)
print('Model loaded. LoRA config ready.')

## Step 6: Dataset + Reward Function

The dataset is curriculum-ordered: warmup papers first (simple numpy-only tasks), then gradually harder papers with MNIST, CartPole, etc.

The reward function combines:
- **Execution reward** — metric proximity + execution quality
- **Format reward** — syntax check + METRICS line detection (no code execution required)

The format reward is critical: it creates reward variance even when all completions fail at runtime, preventing dead batches (zero GRPO gradient).

In [ ]:
import json
import re
from datasets import Dataset
from ScientificLoop.paper_corpus import EVAL_PAPERS, format_paper_for_agent, load_paper, sample_paper
from ScientificLoop.server.execution_engine import compute_metric_proximity, extract_metrics, run_code
from ScientificLoop.reward_calculator import compute_format_reward, compute_step_reward

SYSTEM_PROMPT = (
    'You are an expert ML engineer. '
    'Read the paper methodology below and implement it as a complete, '
    'self-contained Python script. Output ONLY executable Python code — '
    'no markdown, no explanation, no code fences. '
    'The script must print its results on the last line as:\n'
    'METRICS: {"metric_name": value}'
)

def make_dataset(total_episodes=TOTAL_EPISODES):
    rows = []
    for ep in range(total_episodes):
        paper = sample_paper(episode_number=ep)
        rows.append({
            'prompt': f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(paper)}',
            'paper_id': paper.paper_id,
            'episode': ep,
            'difficulty': paper.difficulty,
        })
    return Dataset.from_list(rows)

def _extract_code(raw):
    raw = raw.strip()
    fence = re.search(r'```(?:python)?\n(.*?)```', raw, re.DOTALL)
    if fence:
        return fence.group(1).strip()
    raw = re.sub(r'```(?:python)?', '', raw)
    return re.sub(r'```', '', raw).strip()

def reward_fn(prompts, completions, **kwargs):
    paper_ids = kwargs.get('paper_id', [None] * len(completions))
    rewards = []
    for code, paper_id in zip(completions, paper_ids):
        paper = load_paper(paper_id)
        code = _extract_code(code)
        stdout, stderr, timed_out = run_code(code, paper.execution_timeout)

        if 'SecurityError' in stderr:   exec_status = 'blocked'
        elif timed_out:                 exec_status = 'timeout'
        elif stderr and not stdout:     exec_status = 'error'
        else:                           exec_status = 'success'

        achieved = extract_metrics(stdout)
        score, _ = compute_metric_proximity(achieved, paper.target_metrics, paper.metric_weights)
        matched = sum(
            1 for m, tv in paper.target_metrics.items()
            if m in achieved and abs(achieved[m] - tv) / max(abs(tv), 1e-6) <= 0.10
        )
        exec_reward = compute_step_reward(
            reproduction_score=score, prev_reproduction_score=0.0,
            execution_status=exec_status, step_number=1,
            current_code=code, previous_code=None,
            metrics_matched_count=matched, total_metrics=len(paper.target_metrics),
        )
        fmt_reward = compute_format_reward(code)
        rewards.append(round(exec_reward + 0.5 * fmt_reward, 4))
    return rewards

train_dataset = make_dataset()
print(f'Dataset: {len(train_dataset)} episodes')
print(f'Example prompt (first 300 chars):\n{train_dataset[0]["prompt"][:300]}...')

## Step 7: GRPO Training

Key settings:
- `num_generations=4` — 4 completions per prompt, compared relative to each other
- `temperature=1.1` — slightly higher than default for more diverse outputs
- `gradient_checkpointing=True` — required to fit 7B model in GPU memory
- `learning_rate=1e-5` — standard for LoRA GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    max_completion_length=1024,
    num_generations=4,
    temperature=1.1,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    report_to='none',
    bf16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=grpo_config,
    peft_config=lora_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f'Starting GRPO training — {TOTAL_EPISODES} episodes x {NUM_EPOCHS} epochs')
print(f'Total steps: {TOTAL_EPISODES // 4 * NUM_EPOCHS}')
trainer.train()

## Step 8: Save the Final Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')
if PUSH_TO_HUB:
    print(f'Also pushed to: https://huggingface.co/{HUB_MODEL_ID}')

## Step 9: Quick Inference Test

Try the trained model on a warmup paper to verify it generates runnable code.

In [ ]:
from ScientificLoop.paper_corpus import load_paper

paper = load_paper('linear_regression_gd')
prompt = f'{SYSTEM_PROMPT}\n\n{format_paper_for_agent(paper)}'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)

generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
code = _extract_code(generated)
print('=== Generated Code ===')
print(code[:1500])
print()

stdout, stderr, timed_out = run_code(code, paper.execution_timeout)
achieved = extract_metrics(stdout)
score, deltas = compute_metric_proximity(achieved, paper.target_metrics, paper.metric_weights)
print(f'=== Execution Results ===')
print(f'stdout: {stdout[:500]}')
if stderr: print(f'stderr: {stderr[:200]}')
print(f'\nReproduction score: {score:.3f}')
print(f'Achieved: {achieved}')
print(f'Target:   {paper.target_metrics}')

## Links

- **Environment (HF Space):** https://huggingface.co/spaces/Sushant0809/scientific-loop
- **Trained Model (LoRA adapter):** https://huggingface.co/Sushant0809/scientific-loop-grpo
- **Blog post:** [ScientificLoop: Teaching an LLM to Reproduce Scientific Papers with RL](https://huggingface.co/spaces/Sushant0809/scientific-loop/blob/main/hf_blog_post.md)

---
*OpenEnv Hackathon — PyTorch × Scaler — April 2026*